# qBraid GPU state vector simulator

> **Billing notice:** each GPU simulator has a **free daily allowance**: 25 jobs a day on the Free plan, 125 on Standard, and unlimited on Pro and Enterprise. Past it, jobs are refused unless you turn on usage-based billing on your Usage page, which charges them at **5 credits per minute** of execution time. This notebook runs about ten jobs.

`qbraid:qbraid:sim:gpu-sv` is qBraid's GPU **state vector** simulator. It runs on NVIDIA GH200
Grace Hopper GPUs and simulates every amplitude, so its results are **exact** for any circuit up
to **35 qubits** (34 in double precision), however entangled.

For wider circuits that stay weakly entangled, use its sibling, the GPU tensor network simulator
`qbraid:qbraid:sim:gpu-tn`, which has its own notebook.

This notebook covers:

1. Connecting and running a first circuit
2. Reading how a result was simulated
3. Single and double precision
4. Reproducible sampling with `seed`
5. A 30-qubit circuit, and how long wide circuits take
6. Time limits, and what a refused option looks like

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[qiskit,visualization]'

## Connect to the device

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
device = provider.get_device("qbraid:qbraid:sim:gpu-sv")

device.metadata()

gpu-sv takes OpenQASM 2 or 3, and qBraid transpiles to it for you, so you can pass a Qiskit,
Cirq, Braket, pyQuil, or pytket circuit straight to `run()`:

In [ ]:
device.supported_run_inputs()

## A first run

A three-qubit GHZ state, written in Qiskit:

In [ ]:
from qiskit import QuantumCircuit


def ghz(num_qubits: int) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits)
    circuit.h(0)
    for qubit in range(num_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.measure_all()
    return circuit


circuit = ghz(3)
circuit.draw()

In [ ]:
job = device.run(circuit, shots=1000)
result = job.result()
result.data.get_counts()

Only `000` and `111`, about half each: the simulation is exact and noiseless. Keys are
little-endian: **qubit 0 is the rightmost bit**.

## How the job was simulated

Each result records how it was simulated in its `details`:

In [ ]:
{key: result.details[key] for key in ("numQubits", "simulationMethod", "precision")}

## Runtime options

gpu-sv's settings go in a **`runtime_options` dict**, not in the `run()` signature. `shots` is
the exception: pass it to `run()` directly, from 1 to 1,000,000.

| Option | Default | What it does |
|---|---|---|
| `precision` | `"single"` | `"single"` (complex64, up to 35 qubits) or `"double"` (complex128, up to 34 qubits) |
| `seed` | none | Seeds the sampling: the same seed, circuit and options return the same counts. 0 to 2³² − 1 |
| `timeout` | none | A time limit in whole seconds, at least 1 |

### Single and double precision

Single precision is the default, and it is what reaches 35 qubits. Double precision halves the
rounding error and costs one qubit of reach:

In [ ]:
double_job = device.run(ghz(3), shots=1000, runtime_options={"precision": "double"})
double_result = double_job.result()

print(double_result.data.get_counts())
print(double_result.details["precision"])

### Reproducible results with `seed`

Sampling is random, so two identical runs normally give different counts. Fix `seed` and they
match exactly. This is useful for tests, and for comparing other options while holding the
sampling fixed:

In [ ]:
seeded = [
    device.run(ghz(3), shots=1000, runtime_options={"seed": 7}).result().data.get_counts()
    for _ in range(2)
]

print(seeded[0])
print(seeded[1])
print("identical:", seeded[0] == seeded[1])

Counts are reproducible on the same simulator release; a later release may sample differently
from the same seed.

## A wider circuit

A state vector doubles in size with every qubit: 30 qubits in single precision is 8 GiB of
amplitudes, which the GPU holds comfortably. A GHZ state keeps the counts easy to read:

In [ ]:
import time

start = time.perf_counter()
wide_job = device.run(ghz(30), shots=1000)
wide_counts = wide_job.result().data.get_counts()
elapsed = time.perf_counter() - start

print({f"{key[:6]}…{key[-6:]}": count for key, count in wide_counts.items()})
print(f"{elapsed:.0f} s end to end")

Most of that time is submission and queueing, not simulation. Past 33 qubits in single
precision, or 32 in double, the state no longer fits in the GPU's memory, and gpu-sv keeps the
rest in the GH200's host memory automatically. Those jobs work, but take noticeably longer.

## Time limits and refused options

`timeout` stops a job that runs too long. gpu-sv checks it between the steps of a job, so a
simulation that is already running can take longer than the limit.

An option the simulator doesn't know, or a value out of range, fails the job instead of being
ignored. The reason is on the job's status, in `status_message`, which is the first place to
look when a job fails. A failed job is not charged:

In [ ]:
bad_job = device.run(ghz(3), shots=1000, runtime_options={"precison": "double"})
bad_job.wait_for_final_state()

status = bad_job.status()
print(status)
print(status.status_message)

The misspelt `precison` is named in the message.

## Summary

- `qbraid:qbraid:sim:gpu-sv` simulates any circuit up to 35 qubits exactly, on an NVIDIA GH200.
- Settings go in `runtime_options`: `precision`, `seed` and `timeout`.
- `result.details` records the method and precision a result was simulated with.
- Each GPU simulator has a free daily allowance; past it, usage-based billing charges 5 credits per minute.
- When a job fails, `job.status().status_message` says why.
- For wide, weakly entangled circuits, use `qbraid:qbraid:sim:gpu-tn`.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>